In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

In [3]:
data_dir = str(PROJECT_ROOT / "data" / "processed" / "TEP") + "/"
models_dir = str(PROJECT_ROOT / "models" / "TEP") + "/"

In [4]:
import os
os.makedirs(models_dir, exist_ok=True)

# 1. Fitting PCA Models

In [5]:
training_file_name = "TEP_Training.parquet"

In [6]:
df = pd.read_parquet(data_dir + training_file_name)

In [7]:
n_faultfree_samples = df[df["faultNumber"] == 0].shape[0]
df_faultfree = df.iloc[:n_faultfree_samples]
n_samples_per_ts = df[df["faultNumber"] == 0].groupby("simulationRun").size().max()
df_faultfree = df_faultfree.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [8]:
process_children = ['X6', 'X7', 'X8', 'X9', 'X11', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X22']
n_process_children = len(process_children)

In [9]:
df_faultfree = df_faultfree[process_children]

pca = PCA(n_components=0.95)
pca.fit(df_faultfree)

,n_components,0.95
,copy,True
,whiten,False
,svd_solver,'auto'
,tol,0.0
,iterated_power,'auto'
,n_oversamples,10
,power_iteration_normalizer,'auto'
,random_state,None


# 2. Select Ablation Threshold

In [10]:
df = pd.read_parquet(data_dir + training_file_name)

In [11]:
n_faulty_samples = df[df["faultNumber"] != 0].shape[0]
df_faulty = df.iloc[-n_faulty_samples:]

n_samples_per_faulty_ts = df[df["faultNumber"] != 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faulty_ts = df[df["faultNumber"] != 0].groupby(["sample"]).size().max()

df_faulty = df_faulty.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [12]:
df_faulty = df_faulty[process_children]

In [13]:
cal_scores = pca.transform(df_faulty)
df_faulty_reconstructed = pca.inverse_transform(cal_scores)
spe_cal = np.sum((df_faulty - df_faulty_reconstructed) ** 2, axis=1)

spe_cal_2d = pd.DataFrame(spe_cal.values.reshape(n_faulty_ts, n_samples_per_faulty_ts))

In [14]:
n_faultfree_samples_per_ts = 20

In [15]:
min_window = 6
max_window = 6
min_consecutive_alarms = 5
max_consecutive_alarms = 5

windows = range(min_window, max_window + 1)
consecutive_alarms_list = range(min_consecutive_alarms, max_consecutive_alarms + 1)

idx_first_anomaly = n_faultfree_samples_per_ts - max_window + 1

y_anomaly_2d_red = np.concatenate((
    np.zeros((n_faulty_ts, n_faultfree_samples_per_ts - max_window + 1), dtype=int),
    np.ones((n_faulty_ts, n_samples_per_faulty_ts - n_faultfree_samples_per_ts), dtype=int)
), axis=1)

y_ts_anomaly = np.concatenate((
    np.zeros((n_faulty_ts, 1), dtype=int),
    np.ones((n_faulty_ts, 1), dtype=int)
), axis=1).reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample Accuracy",
    "Sample TPR",
    "Sample FPR",
    "Sample F1",
    "Trajectory-Phase Accuracy",
    "Trajectory-Phase TPR",
    "Trajectory-Phase FPR",
    "Trajectory-Phase F1",
    "Median Anomaly Detection Delay"
])

for w in windows:

    spe_cal_2d_smoothed = spe_cal_2d.T.rolling(window=w).mean().values.T

    spe_cal_2d_smoothed = spe_cal_2d_smoothed[:, max_window - 1:]

    max_spe_cal_smoothed = spe_cal_2d_smoothed[:, : idx_first_anomaly].max(axis=1)
    threshold = np.max(max_spe_cal_smoothed)

    raw_breaches = (spe_cal_2d_smoothed > threshold).astype(int)

    for c in consecutive_alarms_list:
        if c > 1:
            y_anomaly_pred_2d = (
                pd.DataFrame(raw_breaches.T).rolling(window=c, min_periods=c).sum() == c
            ).astype(int).values.T
            
            y_anomaly_pred_2d = np.nan_to_num(y_anomaly_pred_2d, nan=0.0)

            eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
            eval_mask_2d[:, :c-1] = False
        else:
            y_anomaly_pred_2d = raw_breaches
            eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)


        y_anomaly = y_anomaly_2d_red.reshape(-1)
        y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
        eval_mask = eval_mask_2d.reshape(-1)

        y_anomaly_eval = y_anomaly[eval_mask]
        y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

        tp = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 1))
        fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
        fn = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 0))
        tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))

        accuracy = (tp + tn) / (tp + fp + fn + tn)
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

        y_ts_anomaly_pred_2d = np.empty((n_faulty_ts, 2))
        y_ts_anomaly_pred_2d[:, 0] = y_anomaly_pred_2d[:, :idx_first_anomaly].max(axis=1)

        faulty_phase_preds = y_anomaly_pred_2d[:, idx_first_anomaly:]
        y_ts_anomaly_pred_2d[:, 1] = faulty_phase_preds.max(axis=1)

        detected_mask = faulty_phase_preds.any(axis=1)
        if np.any(detected_mask):
            idx_first_anomaly_pred = faulty_phase_preds[detected_mask].argmax(axis=1)
            median_idx_first_anomaly_pred = np.median(idx_first_anomaly_pred)
        else:
            median_idx_first_anomaly_pred = np.nan
        
        y_ts_anomaly_pred = y_ts_anomaly_pred_2d.reshape(-1)

        tp_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 1))
        fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
        fn_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 0))
        tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

        accuracy_ts = (tp_ts + tn_ts) / (tp_ts + fp_ts + fn_ts + tn_ts)
        tpr_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
        fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0
        precision_ts = tp_ts / (tp_ts + fp_ts) if (tp_ts + fp_ts) > 0 else 0
        recall_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
        f1_ts = 2 * precision_ts * recall_ts / (precision_ts + recall_ts) if precision_ts + recall_ts > 0 else 0

        df_result = pd.DataFrame({
            "Window Size": [w],
            "Consecutive Alarms": [c],
            "Threshold": [threshold],
            "Sample Accuracy": [accuracy],
            "Sample TPR": [tpr],
            "Sample FPR": [fpr],
            "Sample F1": [f1],
            "Trajectory-Phase Accuracy": [accuracy_ts],
            "Trajectory-Phase TPR": [tpr_ts],
            "Trajectory-Phase FPR": [fpr_ts],
            "Trajectory-Phase F1": [f1_ts],
            "Median Anomaly Detection Delay": [median_idx_first_anomaly_pred],
        })

        df_results = pd.concat([df_results, df_result], ignore_index=True)

best_idx = df_results["Trajectory-Phase F1"].idxmax()
best_window = df_results['Window Size'][best_idx]
best_cons_alarms = df_results['Consecutive Alarms'][best_idx]
best_threshold = df_results['Threshold'][best_idx]

print("Highest Trajectory-Phase F1:")
print(f"\tWindow Size = {best_window}")
print(f"\tConsecutive Alarms = {best_cons_alarms}")
print(f"\tThreshold = {best_threshold}")

with pd.option_context('display.max_rows', None):
    display(df_results.sort_values("Trajectory-Phase F1", ascending=False))

Highest Trajectory-Phase F1:
	Window Size = 6
	Consecutive Alarms = 5
	Threshold = 1.1396991258507836


<ipykernel>\1271020241.py:123: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Median Anomaly Detection Delay
0,6,5,1.139699,0.457413,0.444979,0.0,0.615896,0.94185,0.8837,0.0,0.93826,28.0


# 3. Test on Faulty Time Series (Ablation Settings)

In [16]:
test_file_name = "TEP_Testing.parquet"

In [17]:
df = pd.read_parquet(data_dir + test_file_name)

In [18]:
n_faulty_samples = df[df["faultNumber"] != 0].shape[0]
df_faulty = df.iloc[-n_faulty_samples:]

n_samples_per_faulty_ts = df[df["faultNumber"] != 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faulty_ts = df[df["faultNumber"] != 0].groupby(["sample"]).size().max()

df_faulty = df_faulty.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [19]:
df_faulty = df_faulty[process_children]

In [20]:
cal_scores = pca.transform(df_faulty)
df_faulty_reconstructed = pca.inverse_transform(cal_scores)
spe_cal = np.sum((df_faulty - df_faulty_reconstructed) ** 2, axis=1)

spe_cal_2d = pd.DataFrame(spe_cal.values.reshape(n_faulty_ts, n_samples_per_faulty_ts))

In [21]:
n_faultfree_samples_per_ts = 160

In [22]:
window = int(best_window)
consecutive_alarms = int(best_cons_alarms)
threshold = best_threshold

idx_first_anomaly = n_faultfree_samples_per_ts - window + 1

y_anomaly_2d_red = np.concatenate((
    np.zeros((n_faulty_ts, n_faultfree_samples_per_ts - window + 1), dtype=int),
    np.ones((n_faulty_ts, n_samples_per_faulty_ts - n_faultfree_samples_per_ts), dtype=int)
), axis=1)

y_ts_anomaly = np.concatenate((
    np.zeros((n_faulty_ts, 1), dtype=int),
    np.ones((n_faulty_ts, 1), dtype=int)
), axis=1).reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample Accuracy",
    "Sample TPR",
    "Sample FPR",
    "Sample F1",
    "Trajectory-Phase Accuracy",
    "Trajectory-Phase TPR",
    "Trajectory-Phase FPR",
    "Trajectory-Phase F1",
    "Median Anomaly Detection Delay"
])

spe_cal_2d_smoothed = spe_cal_2d.T.rolling(window=window).mean().values.T

spe_cal_2d_smoothed = spe_cal_2d_smoothed[:, window - 1:]

raw_breaches = (spe_cal_2d_smoothed > threshold).astype(int)

y_anomaly_pred_2d = (
    pd.DataFrame(raw_breaches.T).rolling(window=consecutive_alarms, min_periods=consecutive_alarms).sum() == consecutive_alarms
).astype(int).values.T

y_anomaly_pred_2d = np.nan_to_num(y_anomaly_pred_2d, nan=0.0)

eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
eval_mask_2d[:, :consecutive_alarms-1] = False

y_anomaly = y_anomaly_2d_red.reshape(-1)
y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
eval_mask = eval_mask_2d.reshape(-1)

y_anomaly_eval = y_anomaly[eval_mask]
y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

tp = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 1))
fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
fn = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 0))
tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))

accuracy = (tp + tn) / (tp + fp + fn + tn)
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

y_ts_anomaly_pred_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_pred_2d[:, 0] = y_anomaly_pred_2d[:, :idx_first_anomaly].max(axis=1)

faulty_phase_preds = y_anomaly_pred_2d[:, idx_first_anomaly:]
y_ts_anomaly_pred_2d[:, 1] = faulty_phase_preds.max(axis=1)

detected_mask = faulty_phase_preds.any(axis=1)
if np.any(detected_mask):
    idx_first_anomaly_pred = faulty_phase_preds[detected_mask].argmax(axis=1)
    median_idx_first_anomaly_pred = np.median(idx_first_anomaly_pred)
else:
    median_idx_first_anomaly_pred = np.nan

y_ts_anomaly_pred = y_ts_anomaly_pred_2d.reshape(-1)

tp_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 1))
fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
fn_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 0))
tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

accuracy_ts = (tp_ts + tn_ts) / (tp_ts + fp_ts + fn_ts + tn_ts)
tpr_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0
precision_ts = tp_ts / (tp_ts + fp_ts) if (tp_ts + fp_ts) > 0 else 0
recall_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
f1_ts = 2 * precision_ts * recall_ts / (precision_ts + recall_ts) if precision_ts + recall_ts > 0 else 0

df_result = pd.DataFrame({
    "Window Size": [window],
    "Consecutive Alarms": [consecutive_alarms],
    "Threshold": [threshold],
    "Sample Accuracy": [accuracy],
    "Sample TPR": [tpr],
    "Sample FPR": [fpr],
    "Sample F1": [f1],
    "Trajectory-Phase Accuracy": [accuracy_ts],
    "Trajectory-Phase TPR": [tpr_ts],
    "Trajectory-Phase FPR": [fpr_ts],
    "Trajectory-Phase F1": [f1_ts],
    "Median Anomaly Detection Delay": [median_idx_first_anomaly_pred],
})

df_results = pd.concat([df_results, df_result], ignore_index=True)

with pd.option_context('display.max_rows', None):
    display(df_results.sort_values("Trajectory-Phase F1", ascending=False))

<ipykernel>\2674998807.py:108: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Median Anomaly Detection Delay
0,6,5,1.139699,0.534729,0.448591,0.008914,0.61863,0.8878,0.9456,0.17,0.893931,28.0


# 4. Test on Fault-Free Time Series (Ablation Settings)

In [23]:
df = pd.read_parquet(data_dir + test_file_name)

In [24]:
n_faultfree_samples = df[df["faultNumber"] == 0].shape[0]
df_faultfree = df.iloc[:n_faultfree_samples]

n_samples_per_faultfree_ts = df[df["faultNumber"] == 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faultfree_ts = df[df["faultNumber"] == 0].groupby(["sample"]).size().max()

df_faultfree = df_faultfree.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [25]:
df_faultfree = df_faultfree[process_children]

In [26]:
cal_scores = pca.transform(df_faultfree)
df_faultfree_reconstructed = pca.inverse_transform(cal_scores)
spe_cal = np.sum((df_faultfree - df_faultfree_reconstructed) ** 2, axis=1)

spe_cal_2d = pd.DataFrame(spe_cal.to_numpy().reshape(n_faultfree_ts, n_samples_per_faultfree_ts))

In [27]:
scores_2d = spe_cal_2d.values
df_scores_2d = pd.DataFrame(scores_2d)

In [28]:
y_anomaly_2d = np.zeros_like(scores_2d, dtype=int)

In [29]:
window = int(best_window)
consecutive_alarms = int(best_cons_alarms)
threshold = best_threshold

y_anomaly_2d_red = y_anomaly_2d[:, window-1:]

y_ts_anomaly = y_anomaly_2d_red.max(axis=1)
y_ts_anomaly = y_ts_anomaly.reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample FPR",
    "Trajectory-Phase FPR",
])

scores_2d_smoothed = df_scores_2d.T.rolling(window=window).mean().values.T
scores_2d_smoothed = scores_2d_smoothed[:, window-1:]

raw_breaches = (scores_2d_smoothed > threshold).astype(int)

y_anomaly_pred_2d = (
    pd.DataFrame(raw_breaches.T).rolling(
        window=consecutive_alarms, min_periods=consecutive_alarms
    ).sum() == consecutive_alarms
).astype(int).values.T

eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
eval_mask_2d[:, :consecutive_alarms-1] = False

y_anomaly = y_anomaly_2d_red.reshape(-1)
y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
eval_mask = eval_mask_2d.reshape(-1)

y_anomaly_eval = y_anomaly[eval_mask]
y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

y_ts_anomaly_pred = y_anomaly_pred_2d.max(axis=1)
y_ts_anomaly_pred = y_ts_anomaly_pred.reshape(-1)

fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))
fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0

df_result = pd.DataFrame({
    "Window Size": [window],
    "Consecutive Alarms": [consecutive_alarms],
    "Threshold": [threshold],
    "Sample FPR": [fpr],
    "Trajectory-Phase FPR": [fpr_ts],
})

df_results = pd.concat([df_results, df_result], ignore_index=True)

with pd.option_context('display.max_rows', None):
    display(df_results)

<ipykernel>\1437690877.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


,Window Size,Consecutive Alarms,Threshold,Sample FPR,Trajectory-Phase FPR
0,6,5,1.139699,0.013533,0.78
